# Credit Card Customer Segmentation

This notebook applies a complete customer-segmentation workflow to credit-card behavioral data, from exploratory analysis and data-quality remediation through K-Means clustering, cluster interpretation, and targeted marketing recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options for pandas DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
# Define the dataset URL
dataset_url = 'https://raw.githubusercontent.com/Lilyrgb/learning-python/refs/heads/main/credit_card_customers.csv'

# Load the dataset into a pandas DataFrame
df = pd.read_csv(dataset_url)

# Display the first 5 rows of the DataFrame
print("\nFirst 5 rows of the dataset:")
display(df.head())

## Exploratory Data Analysis (EDA)



In [ ]:
# Check data types and non-null values
print("\nData types and non-null values:")
df.info()

# Generate descriptive statistics
print("\nDescriptive statistics:")
display(df.describe())

# Check for missing values
print("\nMissing values per column:")
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing Percentage': missing_percentage})
display(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False))

# Check for duplicate rows
num_duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {num_duplicates}")



## Distribution Analysis

Visualising the distribution of key numeric features to better understand the data before cleaning and preprocessing.

In [ ]:
# Histograms of monetary features
monetary_features = [
    'BALANCE', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES',
    'CASH_ADVANCE', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS'
]

plt.figure(figsize=(14, 8))

for i, col in enumerate(monetary_features):
    plt.subplot(2, 4, i + 1)
    plt.hist(df[col].dropna(), bins=40, color='steelblue')
    plt.title(col, fontsize=8)

plt.suptitle('Monetary Features - Distributions', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Histograms of frequency features
frequency_features = [
    'BALANCE_FREQUENCY', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
    'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'PRC_FULL_PAYMENT'
]

plt.figure(figsize=(12, 6))

for i, col in enumerate(frequency_features):
    plt.subplot(2, 3, i + 1)
    plt.hist(df[col].dropna(), bins=30, color='orange')
    plt.axvline(x=1.0, color='red', linestyle='--', label='max=1.0')
    plt.title(col, fontsize=8)
    plt.legend(fontsize=7)

plt.suptitle('Frequency Features - Distributions', fontsize=12)
plt.tight_layout()
plt.show()

## Data Quality Checks & Anomaly Fixes

Before preprocessing, all structural inconsistencies detected in the dataset are corrected.

**High Severity**

1. `BALANCE > CREDIT_LIMIT` (227 rows) — over-limit is a real financial state; `OVER_LIMIT_FLAG` binary feature added.
2. `MINIMUM_PAYMENTS > PAYMENTS` (2,365 rows) — valid revolving debt; `PAYMENT_RATIO = PAYMENTS / MINIMUM_PAYMENTS` added (capped at 10).
3. `MINIMUM_PAYMENTS > CREDIT_LIMIT` (346 rows) — accumulated arrears; handled via `PAYMENT_RATIO`.

**Medium Severity**

4. `CASH_ADVANCE_FREQUENCY > 1` (8 rows) — clipped to 1.0.
5. `PURCHASES ≠ ONEOFF + INSTALLMENTS` (19 rows) — `PURCHASES` replaced with component sum.

**Low Severity**

6. `PURCHASES = 0` but `PURCHASES_TRX > 0` (3 rows) — `PURCHASES_TRX` set to 0.
7. `PURCHASES_TRX = 0` but `PURCHASES_FREQUENCY > 0` (4 rows) — `PURCHASES_FREQUENCY` set to 0.


In [ ]:
# DATA QUALITY CHECKS

line_width = 60

print("=" * line_width)
print("1. BASIC STATISTICS")
print("=" * line_width)
display(df.describe().T[['min', '25%', '50%', '75%', 'max', 'mean']])

print("\n" + "=" * line_width)
print("2. MISSING VALUES")
print("=" * line_width)
missing = df.isnull().sum()
print(missing[missing > 0])

print("\n" + "=" * line_width)
print("3. DUPLICATE CUST_IDs")
print("=" * line_width)
print(f"Duplicate IDs: {df['CUST_ID'].duplicated().sum()}")

print("\n" + "=" * line_width)
print("4. FREQUENCY SCORES OUT OF RANGE [0, 1]")
print("=" * line_width)
freq_cols = [
    'BALANCE_FREQUENCY', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
    'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'PRC_FULL_PAYMENT'
]
for col in freq_cols:
    n = (df[col] > 1).sum()
    if n > 0:
        print(f" {col} > 1 → {n} rows")
    else:
        print(f" {col} OK")

print("\n" + "=" * line_width)
print("5. PURCHASES vs ONEOFF + INSTALLMENTS")
print("=" * line_width)
diff = (df['PURCHASES'] - (df['ONEOFF_PURCHASES'] + df['INSTALLMENTS_PURCHASES'])).abs()
n = (diff > 1.0).sum()
print(f"Rows where |PURCHASES - (ONEOFF + INSTALL)| > 1 : {n}")

print("\n" + "=" * line_width)
print("6. ZERO AMOUNT BUT POSITIVE TRANSACTION COUNT")
print("=" * line_width)
n1 = ((df['PURCHASES'] == 0) & (df['PURCHASES_TRX'] > 0)).sum()
n2 = ((df['CASH_ADVANCE'] == 0) & (df['CASH_ADVANCE_TRX'] > 0)).sum()
print(f"PURCHASES=0 but PURCHASES_TRX>0  : {n1} rows")
print(f"CASH_ADVANCE=0 but CASH_ADV_TRX>0: {n2} rows")

print("\n" + "=" * line_width)
print("7. ZERO TRANSACTION COUNT BUT POSITIVE FREQUENCY")
print("=" * line_width)
n3 = ((df['PURCHASES_TRX'] == 0) & (df['PURCHASES_FREQUENCY'] > 0)).sum()
print(f"PURCHASES_TRX=0 but PURCHASES_FREQUENCY>0: {n3} rows")

print("\n" + "=" * line_width)
print("8. BALANCE > CREDIT_LIMIT")
print("=" * line_width)
n4 = (df['BALANCE'] > df['CREDIT_LIMIT']).sum()
print(f"Rows where BALANCE > CREDIT_LIMIT: {n4}")

print("\n" + "=" * line_width)
print("9. MINIMUM_PAYMENTS > PAYMENTS")
print("=" * line_width)
n5 = (df['MINIMUM_PAYMENTS'] > df['PAYMENTS']).sum()
print(f"Rows where MINIMUM_PAYMENTS > PAYMENTS: {n5} ({n5/len(df)*100:.1f}%)")

print("\n" + "=" * line_width)
print("10. MINIMUM_PAYMENTS > CREDIT_LIMIT")
print("=" * line_width)
n6 = (df['MINIMUM_PAYMENTS'] > df['CREDIT_LIMIT']).sum()
print(f"Rows where MINIMUM_PAYMENTS > CREDIT_LIMIT: {n6}")

In [ ]:
# DATA QUALITY FIXES

# Make a copy of the DataFrame to preserve the original for potential future use
df_clean = df.copy()

# FIX 1: CASH_ADVANCE_FREQUENCY must be in [0, 1] — clip values above 1
n_fix1 = (df_clean['CASH_ADVANCE_FREQUENCY'] > 1).sum()
df_clean['CASH_ADVANCE_FREQUENCY'] = df_clean['CASH_ADVANCE_FREQUENCY'].clip(upper=1.0)
print(f'FIX 1 | CASH_ADVANCE_FREQUENCY clipped to 1.0 → {n_fix1} rows corrected')

# FIX 2: PURCHASES should equal ONEOFF_PURCHASES + INSTALLMENTS_PURCHASES
component_sum = df_clean['ONEOFF_PURCHASES'] + df_clean['INSTALLMENTS_PURCHASES']
mask_fix2 = (component_sum - df_clean['PURCHASES']).abs() > 1.0
n_fix2 = mask_fix2.sum()
df_clean.loc[mask_fix2, 'PURCHASES'] = component_sum[mask_fix2]
print(f'FIX 2 | PURCHASES replaced with ONEOFF+INSTALL sum → {n_fix2} rows corrected')

# FIX 3: PURCHASES = 0 but PURCHASES_TRX > 0 — reset TRX counter
mask_fix3 = (df_clean['PURCHASES'] == 0) & (df_clean['PURCHASES_TRX'] > 0)
n_fix3 = mask_fix3.sum()
df_clean.loc[mask_fix3, 'PURCHASES_TRX'] = 0
print(f'FIX 3 | PURCHASES_TRX set to 0 where PURCHASES=0 → {n_fix3} rows corrected')

# FIX 4: PURCHASES_TRX = 0 but PURCHASES_FREQUENCY > 0 — align to TRX counter
mask_fix4 = (df_clean['PURCHASES_TRX'] == 0) & (df_clean['PURCHASES'] == 0) & (df_clean['PURCHASES_FREQUENCY'] > 0)
n_fix4 = mask_fix4.sum()
df_clean.loc[mask_fix4, 'PURCHASES_FREQUENCY'] = 0.0
print(f'FIX 4 | PURCHASES_FREQUENCY set to 0 where TRX=0 and PURCHASES=0 → {n_fix4} rows corrected')

# FIX 5: BALANCE > CREDIT_LIMIT — add binary flag as engineered feature
df_clean['OVER_LIMIT_FLAG'] = (df_clean['BALANCE'] > df_clean['CREDIT_LIMIT']).astype(int)
n_fix5 = df_clean['OVER_LIMIT_FLAG'].sum()
print(f'FIX 5 | OVER_LIMIT_FLAG added (1 = balance exceeds credit limit) → {n_fix5} customers flagged')

# FIX 6 & 7: MINIMUM_PAYMENTS > PAYMENTS / > CREDIT_LIMIT
# MINIMUM_PAYMENTS represents accumulated arrears — keep as-is and add PAYMENT_RATIO
# NaN imputation of MINIMUM_PAYMENTS is handled in the preprocessing cell below
# Here we use a temporary fill only to compute the ratio without errors
min_pay_temp = df_clean['MINIMUM_PAYMENTS'].fillna(df_clean['MINIMUM_PAYMENTS'].median())
df_clean['PAYMENT_RATIO'] = (df_clean['PAYMENTS'] / min_pay_temp).clip(upper=10.0).round(4)
n_fix67 = (df_clean['PAYMENT_RATIO'] < 1).sum()
print(f'FIX 6/7 | PAYMENT_RATIO added (PAYMENTS / MIN_PAYMENTS, capped at 10) → {n_fix67} customers paying below minimum')

print('\n── Summary ─────────────────────────────────────────────────')
print(f'Original columns   : {df.shape[1]}')
print(f'Columns after fixes: {df_clean.shape[1]} (+OVER_LIMIT_FLAG, +PAYMENT_RATIO)')
print(f'Remaining NaNs     : {df_clean.isnull().sum().sum()} (MINIMUM_PAYMENTS and CREDIT_LIMIT — imputed in next cell)')

## Data Preprocessing

Based on the initial EDA, the following preprocessing steps are required:

1.  Handle Missing Values: Impute missing values in `MINIMUM_PAYMENTS` and `CREDIT_LIMIT` with the median.
2.  Drop `CUST_ID`: Remove the unique customer identifier as it's not relevant for clustering.
3.  Feature Scaling: Standardize all numerical features using `StandardScaler` to ensure that all features contribute equally to the distance calculations in clustering algorithms.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Work on the already quality-fixed DataFrame
df_processed = df_clean.copy()

# 1. Handle Missing Values
# Impute 'MINIMUM_PAYMENTS' with its median
median_minimum_payments = df_processed['MINIMUM_PAYMENTS'].median()
df_processed['MINIMUM_PAYMENTS'] = df_processed['MINIMUM_PAYMENTS'].fillna(median_minimum_payments)
print(f"Missing values in 'MINIMUM_PAYMENTS' imputed with median: {median_minimum_payments:.2f}")

# Impute 'CREDIT_LIMIT' with its median
median_credit_limit = df_processed['CREDIT_LIMIT'].median()
df_processed['CREDIT_LIMIT'] = df_processed['CREDIT_LIMIT'].fillna(median_credit_limit)
print(f"Missing values in 'CREDIT_LIMIT' imputed with median: {median_credit_limit:.2f}")

# Recompute PAYMENT_RATIO now that MINIMUM_PAYMENTS has no NaNs
df_processed['PAYMENT_RATIO'] = (df_processed['PAYMENTS'] / df_processed['MINIMUM_PAYMENTS']).clip(upper=10.0).round(4)
print("PAYMENT_RATIO recomputed on imputed MINIMUM_PAYMENTS.")

# 2. Drop CUST_ID
df_processed.drop('CUST_ID', axis=1, inplace=True)
print("\n'CUST_ID' column dropped.")

# Verify no more missing values remain
assert df_processed.isnull().sum().sum() == 0, "ERROR: missing values still present!"
print("No missing values remaining — OK")

# 3. Feature Scaling
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_processed),
    columns=df_processed.columns
)

print("\nData successfully scaled using StandardScaler.")
print("\nFirst 5 rows of the scaled dataset:")
display(df_scaled.head())

## Clustering Segmentation

The first step is to determine the optimal number of clusters using the Elbow Method with K-Means. Then, calculate the Silhouette Score per $k$ to provide quantitative confirmation of clustering quality.

In [ ]:
from sklearn.cluster import KMeans

# Determine the optimal number of clusters using the Elbow Method
wcss = []
# Test K-values from 1 to 10 (or a reasonable range based on problem)
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10) # n_init to suppress warning
    kmeans.fit(df_scaled)
    wcss.append(kmeans.inertia_)

# Plot the Elbow Method graph
plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method to Determine Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.grid(True)
plt.show()

print("The Elbow Method plot is displayed above. Please examine the plot to identify the 'elbow' point, which suggests the optimal number of clusters for K-Means.")

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
# Test K-values from 2 to 10 (Silhouette Score is not defined for K=1)
for i in range(2, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    score = silhouette_score(df_scaled, kmeans.labels_)
    silhouette_scores.append(score)

# Plot the Silhouette Score graph
plt.figure(figsize=(8, 4))
plt.plot(range(2, 11), silhouette_scores, marker='o', linestyle='--')
plt.title('Silhouette Score to Determine Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

print("The Silhouette Score plot is displayed above. Look for the peak in the plot, which suggests a potentially optimal number of clusters.")

After reviewing both the Elbow Method and the Silhouette Score, we can make a more informed decision on the optimal number of clusters.

In [ ]:
# We will proceed with K=4 for this analysis.
optimal_k = 4

# Apply K-Means clustering with the chosen K
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
clusters = kmeans.fit_predict(df_scaled)

# Add the cluster labels to the original DataFrame for interpretation
# Use df_processed (which has CUST_ID dropped) for easier interpretation of values
df_clustered = df_processed.copy()
df_clustered['CLUSTER'] = clusters + 1

## Cluster Interpretation

Analyzing the mean values for each cluster to identify the key characteristics and behaviors of each segment.


In [ ]:
# Analyze the characteristics of each cluster
cluster_summary = df_clustered.groupby('CLUSTER').mean().round(2)
display(cluster_summary.T.style.format("{:.2f}"))

print("\nCluster summary displayed above. Each column represents a cluster, and the rows show the mean value of each feature. This helps in understanding the unique characteristics of each customer segment.")

# Visualize cluster characteristics (using bar plots for key features)
sns.set_style("whitegrid") # Set a style
fig, ax = plt.subplots(figsize=(10, 8)) # Create the plot
cluster_summary.T.plot(kind='barh', ax=ax, width=0.8) # Changed to horizontal bar plot
plt.title('Average Feature Values per Cluster', fontsize=14, pad=15)
plt.xlabel('Mean Value')
plt.ylabel('Features')
# Refine the layout
plt.grid(True)
plt.legend(title='Clusters', loc='best')
plt.tight_layout()
plt.show()

# Calculate the percentage of customers in each cluster
cluster_percentages = df_clustered['CLUSTER'].value_counts(normalize=True) * 100
cluster_percentages = cluster_percentages.sort_index()

print("\nPercentage of Customers per Cluster:")
display(cluster_percentages.to_frame(name='Percentage'))

## Development of Marketing Strategies

Based on the cluster analysis, we can now develop targeted marketing strategies for each customer segment. This approach allows us to optimize the effectiveness of campaigns and personalize the offering of products and services.

In [ ]:
# In-depth analysis and marketing strategies for each cluster

print("Cluster Summary: Feature Distribution")
display(cluster_summary.style.format("{:.2f}"))

cluster_profiles = {
    1: {
        "title": "High Spenders / Responsible Payers",
        "profile": "Very high purchase activity, high credit limits, and strong payment behavior. These customers represent a small but high-value segment.",
        "objective": "Retain high-value customers and cross-sell premium financial products.",
        "actions": [
            "Enroll customers in exclusive loyalty and VIP reward tiers.",
            "Offer carefully evaluated credit-limit increases.",
            "Cross-sell insurance, investment, and premium-card products."
        ]
    },
    2: {
        "title": "Low Users",
        "profile": "Low activity across most metrics, with minimal purchases and cash advances. This segment may include new, inactive, or under-engaged customers.",
        "objective": "Increase initial engagement and card activation.",
        "actions": [
            "Offer introductory cashback on first or renewed transactions.",
            "Run educational campaigns about card benefits and security.",
            "Promote low-risk incentives for small initial purchases."
        ]
    },
    3: {
        "title": "Cash Advance Users / Minimum Payers",
        "profile": "Frequent cash withdrawals, high balances, limited purchase activity, and a tendency to make minimum payments.",
        "objective": "Mitigate credit risk and support healthier repayment behavior.",
        "actions": [
            "Offer personalized debt-consolidation plans.",
            "Provide preferential rates for eligible balance transfers.",
            "Share financial-wellness resources and repayment incentives."
        ]
    },
    4: {
        "title": "Consistent Installment Buyers",
        "profile": "A strong preference for installment purchases, frequent purchase activity, and reliable payment behavior.",
        "objective": "Increase transaction frequency and average purchase value.",
        "actions": [
            "Award targeted points for high-value installment purchases.",
            "Develop merchant-specific installment partnerships.",
            "Use limited-time offers to encourage complementary one-off spending."
        ]
    }
}

for cluster_id, data in cluster_profiles.items():
    percentage = cluster_percentages.loc[cluster_id]

    print("-" * 80)
    print(f"CLUSTER {cluster_id}: {data['title'].upper()} ({percentage:.1f}%)")
    print("-" * 80)

    print("PROFILE:")
    print(f"   {data['profile']}")

    print("\nSTRATEGY:")
    print(f"   Objective: {data['objective']}")

    print("\nACTION PLAN:")
    for action in data['actions']:
        print(f"   - {action}")
    print()

print("-" * 80)
print("End of Analysis")


## Customer Segmentation Overview

In [ ]:
# Define cluster descriptions
cluster_descriptions = {
    1: "Low Users",
    2: "High Spenders / Responsible Payers",
    3: "Cash Advance Users / Minimum Payers",
    4: "Consistent Installment Buyers"
}

# Create labels with cluster number, percentage, and description
pie_labels = []
for i, p in enumerate(cluster_percentages.values):
    cluster_num = cluster_percentages.index[i]
    description = cluster_descriptions.get(cluster_num, "")
    pie_labels.append(f'Cluster {cluster_num} ({p:.2f}%) - {description}')

# Create a pie chart for cluster size distribution
plt.figure(figsize=(10, 6))
plt.pie(cluster_percentages.values, labels=pie_labels, autopct='', startangle=90, colors=sns.color_palette('tab10', len(cluster_percentages)))
plt.title('Distribution of Cluster Sizes')
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

## Conclusions and Final Insight

This project applied **K-Means clustering** to segment 8,950 credit card customers into 4 distinct behavioral groups, following a structured data science pipeline.

### Work Summary

1. **EDA** — explored distributions, identified right-skewed monetary variables and 2 features with missing values (`MINIMUM_PAYMENTS`: 313 NaN, `CREDIT_LIMIT`: 1 NaN).
2. **Data Quality Fixes** — corrected 7 structural anomalies including frequency scores above 1, purchase amount inconsistencies, and conflicting transaction counters. Added 2 engineered features: `OVER_LIMIT_FLAG` and `PAYMENT_RATIO`.
3. **Preprocessing** — imputed missing values with the median, dropped `CUST_ID`, and standardised all features with `StandardScaler`.
4. **Clustering** — selected **k=4** based on the Elbow Method and Silhouette Score (best score at k=3: 0.240; k=4: 0.197). K-Means was fitted with `n_init=10, random_state=42`.

### Cluster Results

| Cluster | Label | Size | Key Behaviour |
|---------|-------|------|---------------|
| 1 | High-Spenders | 4.7% | Very high purchases, high credit limit, frequent buyer |
| 2 | Low-Users | 44.7% | Low activity, minimal purchases, low credit limit |
| 3 | Cash Advance Users | 13.4% | High cash advances, low purchases, high revolving debt |
| 4 | Consistent Installment Buyers | 37.2% | Regular purchases, strong installment usage, reliable payer |

### Key Takeaways

- The largest segment (44.7%) consists of **inactive customers** — a primary target for reactivation campaigns.
- **Cash Advance Users** represent a high-risk group that benefits most from debt consolidation offers and financial advisory services.
- **High-Spenders**, though only 4.7% of the base, likely generate a disproportionate share of revenue and should be retained through loyalty programmes.
- The two engineered features (`OVER_LIMIT_FLAG`, `PAYMENT_RATIO`) added meaningful risk signal to the clustering model, improving segment separability.

### Business Recommendations
Based on the identified clusters, we recommend the following strategic actions:
* **Targeted Marketing:** Create personalized offers for high-value customers to maintain loyalty.
* **Credit Strategy:** Implement tailored credit limit adjustments for users with high utilization rates.
* **Retention Campaigns:** Develop specific initiatives to encourage installment-focused users to increase their transaction frequency.